# Autoencoder implementation

This notebook implements autoencoder using 1 year hourly T_2M data 

In [8]:
from pathlib import Path
import sys
import importlib
import numpy as np

repo_root = Path.cwd().resolve()
for candidate in [repo_root, *repo_root.parents]:
    if (candidate / 'src' / 'my_ml_zoo').exists():
        repo_root = candidate
        break
else:
    raise RuntimeError('Unable to locate repository root containing src/my_ml_zoo')

src_path = repo_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

importlib.invalidate_caches()
for module_name in [
    'my_ml_zoo.data',
    'my_ml_zoo',
    'my_ml_zoo.data.dataset_config',
    'my_ml_zoo.data.file_discovery',
    'my_ml_zoo.data.xarray_dataset',
]:
    if module_name in sys.modules:
        del sys.modules[module_name]

from my_ml_zoo.data import (
    DatasetConfig,
    discover_variable_files,
    select_variable_file,
    open_variable_dataset,
    infer_file_year,
)


### 1. Define dataset 'IAEVALL03'

In [2]:
config_path = repo_root / 'configs' / 'datasets' / 'IAEVALL03.json'
dataset_config = DatasetConfig.load_from_json(config_path)

### 2. Extract the T_2M data for the year 2002

In [3]:
ds_t2m_2000 = open_variable_dataset(dataset_config, 'T_2M', year=2000)['T_2M']
ds_t2m_2000 = ds_t2m_2000.squeeze('height')

n_time, nx, ny = ds_t2m_2000.shape
print('( n_time, nx, ny ) = (', n_time, ',', nx, ',', ny, ')')

( n_time, nx, ny ) = ( 8784 , 412 , 424 )


### 3. Divide the data to:
1. training (70%)
2. validation (15%)
3. test (15%)

In [4]:
n_train = int(0.70 * n_time)
n_val = int(0.15 * n_time)
n_test = n_time - n_train - n_val

train_da = ds_t2m_2000.isel(time=slice(0, n_train))
val_da   = ds_t2m_2000.isel(time=slice(n_train, n_train + n_val))
test_da  = ds_t2m_2000.isel(time=slice(n_train + n_val, n_time))

print('train da shape:',train_da.shape)
print('val da shape:',val_da.shape)
print('test da shape:',test_da.shape)

train da shape: (6148, 412, 424)
val da shape: (1317, 412, 424)
test da shape: (1319, 412, 424)


### 4. Compute the normalization characteristics on the training data

In [11]:
train_mean = train_da.astype(np.float64).mean().item()
train_std = train_da.astype(np.float64).std().item()

print("train_mean =", train_mean)
print("train_std  =", train_std)

train_mean = 285.71655375833683
train_std  = 10.69642451061396


### 5. Normalize the training, validation and test data using the obtained characteristics

In [12]:
train_norm = (train_da - train_mean) / train_std
val_norm   = (val_da   - train_mean) / train_std
test_norm  = (test_da  - train_mean) / train_std

In [14]:
# validation
print("train norm mean:", train_norm.astype(np.float64).mean().item())
print("train norm std :", train_norm.astype(np.float64).std().item())
print("val   norm mean:", val_norm.astype(np.float64).mean().item())
print("val   norm std :", val_norm.astype(np.float64).std().item())

train norm mean: 9.57307441510677e-08
train norm std : 1.0000000024538502
val   norm mean: 0.09319822115502278
val   norm std : 0.7435435963924072
